In [ ]:
!pip install datasets pandas huggingface_hub

In [ ]:
from datasets import load_dataset, DatasetDict
import pandas as pd

reporting = load_dataset("JayShah07/multi_label_reporting")

print("Dataset structure:")
print(reporting)
print("\nFirst training example:")
print(reporting["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/9.93k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/260 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/112 [00:00<?, ? examples/s]

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['user_query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module'],
        num_rows: 260
    })
    validation: Dataset({
        features: ['user_query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module'],
        num_rows: 112
    })
})

First training example:
{'user_query': 'Provide a detailed report of my current holdings.', 'holdings': 1, 'capital_gains': 0, 'scheme_wise_returns': 0, 'investment_account_wise_returns': 0, 'portfolio_update': 0, 'Current Year': 0, 'Previous Year': 0, 'Daily': 0, 'Monthly': 0, 'Weekly': 0, 'Yearly': 0, 'None_date': 1, 'None_module': 0}


In [ ]:
from datasets import concatenate_datasets

# Combine train and validation datasets
combined = concatenate_datasets([reporting["train"], reporting["validation"]])

# Convert to pandas DataFrame
df = combined.to_pandas()

# Rename user_query → query
df = df.rename(columns={"user_query": "query"})

# Identify label columns (all except query)
label_cols = [col for col in df.columns if col != "query"]

# Create a list of active labels for each row
df["labels"] = df.apply(lambda row: [col for col in label_cols if row[col] == 1], axis=1)

# OPTIONAL: Keep only query + labels
# df = df[["query", "labels"]]

# Save CSV
df.to_csv("reporting_combined.csv", index=False)

df.head()

,query,holdings,capital_gains,scheme_wise_returns,investment_account_wise_returns,portfolio_update,Current Year,Previous Year,Daily,Monthly,Weekly,Yearly,None_date,None_module,labels
0,Provide a detailed report of my current holdings.,1,0,0,0,0,0,0,0,0,0,0,1,0,"[holdings, None_date]"
1,Can you send me a report of my holdings as of ...,1,0,0,0,0,0,0,0,0,0,0,1,0,"[holdings, None_date]"
2,Provide a report on my daily returns.,0,0,0,0,1,0,0,1,0,0,0,0,0,"[portfolio_update, Daily]"
3,Can you send me a report of my holdings as of ...,1,0,0,0,0,0,0,0,0,0,0,1,0,"[holdings, None_date]"
4,How has my portfolio performed over the last s...,0,0,0,0,1,0,0,0,1,0,0,0,0,"[portfolio_update, Monthly]"


In [ ]:
new_df = pd.read_csv("/content/synthetic_3500.csv")

new_df.head()

,holdings,capital_gains,scheme_wise_returns,investment_account_wise_returns,portfolio_update,Current Year,Previous Year,Daily,Monthly,Weekly,Yearly,None_date,None_module,query,labels
0,0,1,0,0,0,0,1,0,0,0,0,0,0,Show capital gains accumulated over time from ...,"['capital_gains', 'Previous Year']"
1,0,0,0,0,1,0,0,0,0,0,0,1,0,Give me my latest portfolio update,"['portfolio_update', 'None_date']"
2,0,0,1,0,0,0,0,0,0,0,0,1,0,Give me returns organized by scheme,"['scheme_wise_returns', 'None_date']"
3,1,0,0,0,0,0,0,0,0,0,0,1,0,Share my holdings report,"['holdings', 'None_date']"
4,0,0,0,0,1,0,0,0,0,1,0,0,0,Give me my latest portfolio update as a weekly...,"['portfolio_update', 'Weekly']"


In [ ]:
import pandas as pd

# 1. Load both dataframes
df1 = pd.read_csv("/content/reporting_combined.csv")         # 383 rows approx
df2 = pd.read_csv("/content/synthetic_3500.csv")    # 3500 rows

# 2. Ensure df2 has the columns in the right order
desired_cols = [
    "query",
    "holdings",
    "capital_gains",
    "scheme_wise_returns",
    "investment_account_wise_returns",
    "portfolio_update",
    "Current Year",
    "Previous Year",
    "Daily",
    "Monthly",
    "Weekly",
    "Yearly",
    "None_date",
    "None_module",
    "labels"
]

# Reorder df2, moving "query" first and keeping labels last
df2 = df2[desired_cols]

# 3. Combine row-wise
combined_df = pd.concat([df1, df2], ignore_index=True)

# 4. Save final output
combined_df.to_csv("final_training_dataset.csv", index=False)

combined_df.head()

,query,holdings,capital_gains,scheme_wise_returns,investment_account_wise_returns,portfolio_update,Current Year,Previous Year,Daily,Monthly,Weekly,Yearly,None_date,None_module,labels
0,Provide a detailed report of my current holdings.,1,0,0,0,0,0,0,0,0,0,0,1,0,"['holdings', 'None_date']"
1,Can you send me a report of my holdings as of ...,1,0,0,0,0,0,0,0,0,0,0,1,0,"['holdings', 'None_date']"
2,Provide a report on my daily returns.,0,0,0,0,1,0,0,1,0,0,0,0,0,"['portfolio_update', 'Daily']"
3,Can you send me a report of my holdings as of ...,1,0,0,0,0,0,0,0,0,0,0,1,0,"['holdings', 'None_date']"
4,How has my portfolio performed over the last s...,0,0,0,0,1,0,0,0,1,0,0,0,0,"['portfolio_update', 'Monthly']"


In [ ]:
import pandas as pd

df = pd.read_csv("final_training_dataset.csv")
df.head()

,query,holdings,capital_gains,scheme_wise_returns,investment_account_wise_returns,portfolio_update,Current Year,Previous Year,Daily,Monthly,Weekly,Yearly,None_date,None_module,labels
0,Provide a detailed report of my current holdings.,1,0,0,0,0,0,0,0,0,0,0,1,0,"['holdings', 'None_date']"
1,Can you send me a report of my holdings as of ...,1,0,0,0,0,0,0,0,0,0,0,1,0,"['holdings', 'None_date']"
2,Provide a report on my daily returns.,0,0,0,0,1,0,0,1,0,0,0,0,0,"['portfolio_update', 'Daily']"
3,Can you send me a report of my holdings as of ...,1,0,0,0,0,0,0,0,0,0,0,1,0,"['holdings', 'None_date']"
4,How has my portfolio performed over the last s...,0,0,0,0,1,0,0,0,1,0,0,0,0,"['portfolio_update', 'Monthly']"


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

len(train_df), len(val_df), len(test_df)

(3097, 387, 388)

In [ ]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module', 'labels', '__index_level_0__'],
        num_rows: 3097
    })
    validation: Dataset({
        features: ['query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module', 'labels', '__index_level_0__'],
        num_rows: 387
    })
    test: Dataset({
        features: ['query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module', 'labels', '__index_level_0__'],
        num_rows: 388
    })
})

In [ ]:
dataset_dict = dataset_dict.remove_columns(["__index_level_0__"])
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module', 'labels'],
        num_rows: 3097
    })
    validation: Dataset({
        features: ['query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module', 'labels'],
        num_rows: 387
    })
    test: Dataset({
        features: ['query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module', 'labels'],
        num_rows: 388
    })
})

In [ ]:
from huggingface_hub import login
login()

In [ ]:
dataset_dict.push_to_hub("JayShah07/reporting_final_dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 47.8kB / 47.8kB            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 12.3kB / 12.3kB            

CommitInfo(commit_url='https://huggingface.co/datasets/JayShah07/reporting_final_dataset/commit/fd7d95ddbc282c260a74667eb81d31223e9153b2', commit_message='Upload dataset', commit_description='', oid='fd7d95ddbc282c260a74667eb81d31223e9153b2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JayShah07/reporting_final_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JayShah07/reporting_final_dataset'), pr_revision=None, pr_num=None)